In [1]:
import pandas as pd

In [2]:

def validation_dataset(landsat='data/validate/landsat_features_validation.csv', terraclimate='data/validate/terraclimate_features_validation.csv', col='Sample Date'):
    '''
    input: 2 validation datasets (CSV files) from EY's Water Quality Prediction Benchmark Notebook
    return: merged pandas dataframe with added bin column (monthly) joined on GPS coordinates and date
    '''
    # read CSVs and Handling Missing Values
    landsat = pd.read_csv(landsat)
    # landsat.fillna(landsat.median(numeric_only=True))
    terraclimate = pd.read_csv(terraclimate)

    # merge features and targets on coordinates and date; convert to datetime
    landsat_terracl = pd.merge(landsat, terraclimate)
    landsat_terracl[col] = pd.to_datetime(landsat_terracl[col], dayfirst=True, errors='coerce')

    # bin according to month
    bins = pd.date_range(start=landsat_terracl[col].min(), end=landsat_terracl[col].max(), freq='ME')
    landsat_terracl['month'] = pd.cut(landsat_terracl[col], bins=len(bins), labels=bins)

    # Handling Missing Values
    val_data = landsat_terracl.fillna(landsat_terracl.median(numeric_only=True))
    print('null values:','\n',val_data.isna().sum())
    val_data.columns = landsat_terracl.columns.str.strip().str.lower()

    print('We will validate Water Quality over the course of',len(val_data[['month']].groupby('month', observed=True).count()),'months.')

    return val_data

In [3]:
df = validation_dataset()

null values: 
 Latitude       0
Longitude      0
Sample Date    0
nir            0
green          0
swir16         0
swir22         0
NDMI           0
MNDWI          0
pet            0
month          0
dtype: int64
We will validate Water Quality over the course of 55 months.


In [4]:
df.head()

,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,month
0,-32.043333,27.822778,2014-09-01,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,161.90001,2014-08-31
1,-33.329167,26.077500,2015-09-16,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,177.60000,2015-08-31
2,-32.991639,27.640028,2015-05-07,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,158.40001,2015-04-30
3,-34.096389,24.439167,2012-02-07,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,130.00000,2012-01-31
4,-32.000556,28.581667,2014-10-01,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,152.50000,2014-09-30
